In [0]:
%sql

SELECT * FROM bakery_pos_develop.bronze.product_product

In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.product_product

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW stg_product_product AS SELECT
    id AS product_id,
    name,
    categ_id[0] AS categ_id,
    categ_id[1] AS categ_name,
    lst_price,
    taxes_id,
    default_code,
    standard_price,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.product_product
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.product_product (
    product_id BIGINT,
    name STRING,
    categ_id INT,
    categ_name STRING,
    lst_price DOUBLE,
    taxes_id ARRAY<STRING>,
    default_code STRING,
    standard_price DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.product_product AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_product_product
) AS source
ON target.product_id = source.product_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    name = source.name,
    categ_id = source.categ_id,
    categ_name = source.categ_name,
    lst_price = source.lst_price,
    taxes_id = source.taxes_id,
    default_code = source.default_code,
    standard_price = source.standard_price,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    product_id,
    name,
    categ_id,
    categ_name,
    lst_price,
    taxes_id,
    default_code,
    standard_price,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.product_id,
    source.name,
    source.categ_id,
    source.categ_name,
    source.lst_price,
    source.taxes_id,
    source.default_code,
    source.standard_price,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );